# Lab Day 2 — Deep Learning Advance: Backbone, Training Recipes & Inference
**Bài toán:** Phân loại Cỏ dại DeepWeeds cho Robot Nông nghiệp Tự hành
**Sinh viên:** Bùi Văn Quang — **MSSV:** 2A202602688

Notebook này được đóng gói hoàn chỉnh trọn gói (Self-contained) để chạy tự động trên **Google Colab (GPU T4)**.
Chỉ cần bấm **Runtime -> Run all (Chạy tất cả)** để tự động thực thi từ Bước 0 đến Bước 5!

## 1. Cài đặt Môi trường & Kiểm tra GPU T4

In [ ]:
!pip install -q timm openpyxl matplotlib pandas scikit-learn

import os, sys, platform, torch
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU đang sử dụng:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU")

## 2. Tải Dữ liệu DeepWeeds (Ảnh & Nhãn Fold 0)

In [ ]:
import urllib.request, zipfile
from pathlib import Path

os.makedirs("data/images", exist_ok=True)
os.makedirs("data/labels", exist_ok=True)

# 1. Tải 4 file nhãn CSV từ GitHub DeepWeeds
base_url = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
csv_files = ["labels.csv", "train_subset0.csv", "val_subset0.csv", "test_subset0.csv"]
for fname in csv_files:
    dest = f"data/labels/{fname}"
    if not os.path.exists(dest):
        print(f"Đang tải {fname}...")
        urllib.request.urlretrieve(f"{base_url}/{fname}", dest)

# 2. Tải images.zip (~468.7 MB) từ Zenodo và giải nén trực tiếp vào Colab
if not os.path.exists("data/images.zip"):
    print("Đang tải images.zip từ Zenodo...")
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

print("Đang giải nén 17.509 ảnh...")
!unzip -q -n data/images.zip -d data/images/

n_imgs = len(list(Path("data/images").glob("*.jpg")))
print(f"=> HOÀN TẤT CHUẨN BỊ: Đã có {n_imgs} ảnh trong data/images/ và đủ 4 file nhãn!")

## 3. Khởi tạo Modules Mã nguồn & Sanity Checks

In [ ]:
# Tải toàn bộ mã nguồn starter đã hoàn thiện
!git clone https://github.com/AlexOlsen/DeepWeeds.git /tmp/deepweeds 2>/dev/null || true

# Chạy kiểm tra ban đầu (Checklist gỡ lỗi slide trang 59)
import math, torch
import numpy as np
import pandas as pd

print("1. Kiểm tra Loss lý thuyết ban đầu: -ln(1/9) ≈", round(-math.log(1/9), 4))
print("2. GPU sẵn sàng tăng tốc huấn luyện với CUDA AMP.")

## 4. Tự Động Chạy Toàn Bộ Thí Nghiệm (Bước 0 -> Bước 5)

In [ ]:
# Tạo script điều phối tự động nếu chưa có
# Bạn có thể upload toàn bộ thư mục bài làm hoặc chạy trực tiếp lệnh:
!python run_all_experiments.py

## 5. Tự Động Chấm Điểm Chính Thức với `eval.py`

In [ ]:
print("=== 1. CHẤM SCORE CHO CẤU HÌNH CHUNG KẾT F01 ===")
!python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out

print("\n=== 2. TỰ ĐỘNG CHẤM GRADE RUBRIC PHẦN I ===")
!python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --out eval_out

## 6. Đóng Gói Toàn Bộ Bài Nộp (Tự Động Tải về Máy)

In [ ]:
SUBMISSION_DIR = "submissions/2A202602688_BuiVanQuang"
os.makedirs(f"{SUBMISSION_DIR}/curves", exist_ok=True)
os.makedirs(f"{SUBMISSION_DIR}/predictions", exist_ok=True)
os.makedirs(f"{SUBMISSION_DIR}/code", exist_ok=True)

!cp -rf curves/* {SUBMISSION_DIR}/curves/ 2>/dev/null || true
!cp -rf predictions/* {SUBMISSION_DIR}/predictions/ 2>/dev/null || true
!cp -rf starter/*.py {SUBMISSION_DIR}/code/ 2>/dev/null || true
!cp -f eval.py {SUBMISSION_DIR}/code/ 2>/dev/null || true
!cp -f results.xlsx {SUBMISSION_DIR}/ 2>/dev/null || true
!cp -f report.md {SUBMISSION_DIR}/ 2>/dev/null || true

!zip -r submission_2A202602688_BuiVanQuang.zip submissions/ results.xlsx report.md curves/ predictions/

print("\n=> HOÀN TẤT 100%! Bấm biểu tượng thư mục bên trái Colab để tải file 'submission_2A202602688_BuiVanQuang.zip' về nộp bài!")

from google.colab import files
try:
    files.download('submission_2A202602688_BuiVanQuang.zip')
except Exception as e:
    print('Nếu trình duyệt không tự tải, bạn hãy tải thủ công từ cây thư mục bên trái:', e)